# Tarrant CAD — PropertyData_R_2025 ETL

Exploration / processing notebook for the Tarrant Appraisal District residential export `PropertyData_R_2025.txt`.

**Source format** (checked on the 2025 file):
- Pipe-delimited (`|`), CRLF line endings, 56 columns, ~659K rows
- Values are space-padded fixed-width inside the pipes (e.g. `|      110000|`)
- One row per account (`Record_Type = AAAA`)
- Sentinel values: deed date `12-31-1900`, deed book/page `0000000`, instrument `0000000000`, year built `0`

Steps: **Load raw → Clean → Type → Profile → Validate → Export (bronze)**.

## 1. Setup

In [2]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

DATA_DIR = Path("../Data")
OUTPUT_DIR = Path("../output")
OUTPUT_DIR.mkdir(exist_ok=True)

INPUT_FILE = DATA_DIR / "PropertyDataSupplemental_R_2025.txt"
print(INPUT_FILE.resolve(), f"{INPUT_FILE.stat().st_size / 1e6:,.1f} MB")

/Users/user2/PTP/ptp-data/tarrant/Data/PropertyDataSupplemental_R_2025.txt 352.9 MB


## 2. Load raw
Read everything as string first so nothing (leading zeros in `Account_Num`, zips, codes) gets mangled by type inference.

In [3]:
raw = pd.read_csv(
    INPUT_FILE,
    sep="|",
    dtype=str,
    keep_default_na=False,   # keep blanks as "" until we clean them ourselves
    encoding="latin-1",      # safe for any stray non-UTF-8 bytes
    quoting=3,               # csv.QUOTE_NONE: owner names / legal descriptions can contain quotes
)
raw.columns = raw.columns.str.strip()
print(raw.shape)
raw.head()

(658409, 28)


,RP,AppraisalYear,AccountNumber,AgentName,AgentNumber,Condition,DepreciationPercent,Latitude,Longitude,EYOC,NBHD,Quality,SiteName,SiteClassCd,SiteClassDescr,PartyId,LastSaleDate,SiteId,SitusCity,NAICSDescr,NAICSCd,PercentComplete,NoticeValue,SubdivisionName,NumberofStories,LandUseCode,Country,AddressType
0,R,2025,00001309,,,Good ...,,32.7591221347,-97.3281275822,1995,4T050C ...,Average ...,FORT WORTH ORIGINAL TOWN-29-32 ...,A1,Residential - Single Family ...,3616171,2016-08-22,100001309,,,,100%,,FORT WORTH ORIGINAL TOWN,2,Residential 4T050 ...,UNITED STATES OF AMERICA,Standard
1,R,2025,00006483,,,Fair ...,,32.762335283,-97.3271676957,1955,2M210D ...,Low ...,FORT WORTH ORIGINAL TOWN 124 1B,A1,Residential - Single Family ...,59570,1900-01-01,80874275,,,,100%,101114,FORT WORTH ORIGINAL TOWN,2,Residential By Square Foot ...,UNITED STATES OF AMERICA,Standard
2,R,2025,00006491,,,Poor ...,,32.7622310968,-97.3271440167,1945,2M210D ...,Low ...,FORT WORTH ORIGINAL TOWN 124 1C1,A1,Residential - Single Family ...,775725,1990-06-01,80874276,,,,100%,60890,FORT WORTH ORIGINAL TOWN,1,Residential By Square Foot ...,UNITED STATES OF AMERICA,Standard
3,R,2025,00006505,OCONNOR & ASSOCIATES,00436,,,32.7621419405,-97.3270950408,,2M210D ...,,FORT WORTH ORIGINAL TOWN Block 124 Lot 1D,C1,Residential - Vacant Land ...,11922,1900-01-01,80874289,,,,0%,,FORT WORTH ORIGINAL TOWN,0,Residential By Square Foot ...,UNITED STATES OF AMERICA,Standard
4,R,2025,00006513,OCONNOR & ASSOCIATES,00436,Fair ...,,32.7619349338,-97.326950484,1955,2M210D ...,Low ...,FORT WORTH ORIGINAL TOWN 124 1F,A1,Residential - Single Family ...,11923,1984-03-10,80874277,,,,100%,141751,FORT WORTH ORIGINAL TOWN,1,Residential By Square Foot ...,UNITED STATES OF AMERICA,Standard


In [4]:
raw.describe(include="all").T

,count,unique,top,freq
RP,658409,1,R,658409
AppraisalYear,658409,1,2025,658409
AccountNumber,658409,658409,00001309,1
AgentName,658409,1092,,489358
AgentNumber,658409,1102,,489358
Condition,658409,7,Average ...,232793
DepreciationPercent,658409,1,,658409
Latitude,658409,632356,00000000000000000000000000000000,529
Longitude,658409,632313,00000000000000000000000000000000,529
EYOC,658409,104,1975,91138


In [ ]:
df = raw.apply(lambda s: s.str.strip())
df = df.replace("", pd.NA)

# First column header is "RP" in the file; values are the property type ("R" = residential)
df = df.rename(columns={"RP": "Property_Type"})

df["Property_Type"].value_counts(dropna=False), df["Record_Type"].value_counts(dropna=False)

## 4. Types
Column groups below follow the existing `ptp-bi/Tarrant/main/lambda_main_csv_script.py` dtype map, extended to every column in the file.

In [ ]:
INT_COLS = [
    "Appraisal_Year", "Num_Special_Dist", "Garage_Capacity", "Num_Bedrooms",
    "Num_Bathrooms", "Year_Built", "Structure_Count",
]
FLOAT_COLS = [
    "Land_Value", "Improvement_Value", "Total_Value", "Appraised_Value",
    "Ag_Value", "Living_Area", "Land_Acres", "Land_SqFt", "Ag_Acres",
]
DATE_COLS = ["Notice_Date", "Deed_Date", "Appraisal_Date"]  # MM-DD-YYYY

for c in INT_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
for c in FLOAT_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce").astype("Float64")
for c in DATE_COLS:
    df[c] = pd.to_datetime(df[c], format="%m-%d-%Y", errors="coerce")

# Everything else stays as string
str_cols = [c for c in df.columns if c not in INT_COLS + FLOAT_COLS + DATE_COLS]
df[str_cols] = df[str_cols].astype("string")

df.dtypes

## 5. Sentinel / placeholder values → null

In [ ]:
df.loc[df["Deed_Date"] == "1900-12-31", "Deed_Date"] = pd.NaT
for c in ["Deed_Book", "Deed_Page", "Instrument_No"]:
    df.loc[df[c].str.fullmatch(r"0+", na=False), c] = pd.NA
df.loc[df["Year_Built"] == 0, "Year_Built"] = pd.NA
df.loc[df["Living_Area"] == 0, "Living_Area"] = pd.NA

# Account_Num must be 8 chars, zero-padded
df["Account_Num"] = df["Account_Num"].str.zfill(8)

df[["Deed_Date", "Deed_Book", "Deed_Page", "Instrument_No", "Year_Built", "Living_Area"]].isna().mean().round(3)

## 6. Profile

In [ ]:
print(f"rows: {len(df):,}   unique accounts: {df['Account_Num'].nunique():,}")
dupes = df[df.duplicated("Account_Num", keep=False)]
print(f"duplicate account rows: {len(dupes):,}")

null_pct = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
null_pct[null_pct > 0]

In [ ]:
df[FLOAT_COLS + INT_COLS].describe().T

In [ ]:
for c in ["Property_Class", "State_Use_Code", "City", "School", "Exemption_Code", "Swimming_Pool_Ind", "ARB_Indicator"]:
    print(f"--- {c}")
    print(df[c].value_counts(dropna=False).head(15).to_string(), "\n")

## 7. Validation
Same rules as the bronze Lambda's pandera schema — rows that fail are kept aside with a reason instead of dropped silently.

In [ ]:
current_year = datetime.now().year

rules = {
    "account_num_len_8":    df["Account_Num"].str.len() == 8,
    "total_value_gt_0":     df["Total_Value"] > 0,
    "improvement_value_gt_0": df["Improvement_Value"] > 0,
    "property_class_len":   df["Property_Class"].str.len().between(1, 2),
    "living_area_gt_0":     df["Living_Area"].isna() | (df["Living_Area"] > 0),
    "year_built_range":     df["Year_Built"].isna() | df["Year_Built"].between(1000, current_year + 1),
    "central_heat_yn":      df["Central_Heat_Ind"].isna() | df["Central_Heat_Ind"].isin(["Y", "N"]),
    "central_air_yn":       df["Central_Air_Ind"].isna() | df["Central_Air_Ind"].isin(["Y", "N"]),
}
checks = pd.DataFrame({k: v.fillna(False).astype(bool) for k, v in rules.items()})

print((~checks).sum().sort_values(ascending=False).to_string())

passed = checks.all(axis=1)
df_success = df[passed].copy()
df_failed = df[~passed].copy()
df_failed["Validation_Notes"] = (~checks[~passed]).apply(lambda r: " | ".join(r.index[r]), axis=1)

print(f"\npassed: {len(df_success):,}   failed: {len(df_failed):,}")
df_failed["Validation_Notes"].value_counts().head(10)

## 8. Export
Parquet keeps the dtypes (needs `pyarrow`); CSV matches what the bronze Lambda writes today.

In [5]:
raw.to_csv(OUTPUT_DIR / "PropertyDataSupplemental_R_2025.csv", index=False)

In [ ]:
stem = INPUT_FILE.stem  # PropertyData_R_2025

df.to_parquet(OUTPUT_DIR / f"{stem}_clean.parquet", index=False)
df_success.to_csv(OUTPUT_DIR / f"{stem}_success.csv", index=False)
df_failed.to_csv(OUTPUT_DIR / f"{stem}_failed.csv", index=False)

sorted(p.name for p in OUTPUT_DIR.iterdir())